# 03. Feature Engineering

Create transformed features, balance scaling, and fraud-focused signal engineering.

In [24]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

DATA_PATH = "../data/raw/creditcard.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)

Dataset shape: (284807, 31)


In [25]:
X = df.drop(columns=["Class"])
y = df["Class"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (284807, 30)
y shape: (284807,)


In [26]:
print("Feature columns:")
print(X.columns.tolist())

print("\nTarget:")
print(y.name)

print("Class in X:", "Class" in X.columns)

Feature columns:
['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount']

Target:
Class
Class in X: False


In [27]:
stratify=y
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (199364, 30)
Validation: (42721, 30)
Test: (42722, 30)


In [28]:
print("Train distribution:")
print(y_train.value_counts(normalize=True))

print("\nValidation distribution:")
print(y_val.value_counts(normalize=True))

print("\nTest distribution:")
print(y_test.value_counts(normalize=True))

Train distribution:
Class
0    0.998275
1    0.001725
Name: proportion, dtype: float64

Validation distribution:
Class
0    0.998268
1    0.001732
Name: proportion, dtype: float64

Test distribution:
Class
0    0.998268
1    0.001732
Name: proportion, dtype: float64


In [29]:
X_train = X_train.copy()
X_val = X_val.copy()
X_test = X_test.copy()

X_train["Amount_log"] = np.log1p(X_train["Amount"])
X_val["Amount_log"] = np.log1p(X_val["Amount"])
X_test["Amount_log"] = np.log1p(X_test["Amount"])

print(X_train[["Amount", "Amount_log"]].head())

         Amount  Amount_log
249927     7.13    2.095561
214082   150.00    5.017280
106005  1302.49    7.172801
58619      4.49    1.702928
191638     4.49    1.702928


In [30]:
scaler = StandardScaler()
scaler.fit(X_train)

X_train_scaled = scaler.transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [31]:
X_train_scaled = pd.DataFrame(
    X_train_scaled,
    columns=X_train.columns,
    index=X_train.index
)

X_val_scaled = pd.DataFrame(
    X_val_scaled,
    columns=X_val.columns,
    index=X_val.index
)

X_test_scaled = pd.DataFrame(
    X_test_scaled,
    columns=X_test.columns,
    index=X_test.index
)
X_train_scaled.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Amount_log
249927,1.257992,-0.005578,0.427819,0.108347,-0.534196,0.425864,-0.489116,0.676802,-0.000318,-0.169658,...,-0.313204,-0.788791,0.067073,-0.596329,-0.947613,0.295700,0.579623,0.252462,-0.321082,-0.639152
214082,0.939713,0.904109,-0.110138,-1.449361,0.840920,0.442776,-0.760782,0.736596,-0.322162,-0.291515,...,0.379249,0.729482,-0.372721,-0.565282,0.945219,-0.887094,-0.183513,-0.173725,0.243475,1.125111
106005,-0.528960,-0.550544,-2.676901,-0.919977,0.331363,-1.496917,-0.573444,1.283049,-0.590546,-1.174846,...,0.467427,-1.973705,-1.756150,0.836216,-0.267770,1.804491,-0.776446,0.689448,4.797594,2.426709
58619,-0.977583,-0.263360,0.619460,-0.403529,-0.551463,1.783666,2.495035,0.037183,0.663671,-0.295311,...,-0.261482,-0.975989,0.231863,1.560567,-1.510087,0.082795,0.027490,0.769753,-0.331514,-0.876241
191638,0.725456,-0.325205,0.129183,0.187309,-1.261520,1.885635,3.011145,-0.159076,0.782832,0.430014,...,0.226255,1.045051,-0.515334,1.269290,-0.160452,1.270985,0.182557,-0.102717,-0.331514,-0.876241


In [32]:
print(X_train_scaled.mean().head())

print(X_train_scaled.std().head())

Time   -1.062086e-16
V1      1.140495e-18
V2     -3.421485e-18
V3      9.337804e-18
V4      4.205576e-18
dtype: float64
Time    1.000003
V1      1.000003
V2      1.000003
V3      1.000003
V4      1.000003
dtype: float64


In [33]:
print("Validation means:")
print(X_val_scaled.mean().head())

print("\nTest means:")
print(X_test_scaled.mean().head())

Validation means:
Time   -0.003969
V1     -0.001360
V2      0.004065
V3      0.000055
V4      0.003296
dtype: float64

Test means:
Time   -0.007873
V1      0.005215
V2      0.004074
V3      0.005795
V4     -0.004768
dtype: float64


In [34]:
print("Target in training features:", "Class" in X_train.columns)
print("Target in validation features:", "Class" in X_val.columns)
print("Target in test features:", "Class" in X_test.columns)

Target in training features: False
Target in validation features: False
Target in test features: False


In [35]:
print("Original:", len(df))
print("Train:", len(X_train))
print("Validation:", len(X_val))
print("Test:", len(X_test))

print(
    "Total:",
    len(X_train) + len(X_val) + len(X_test)
)

Original: 284807
Train: 199364
Validation: 42721
Test: 42722
Total: 284807


In [36]:
train_indices = set(X_train.index)
val_indices = set(X_val.index)
test_indices = set(X_test.index)

print("Train ∩ Validation:", len(train_indices & val_indices))
print("Train ∩ Test:", len(train_indices & test_indices))
print("Validation ∩ Test:", len(val_indices & test_indices))

Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0
